# Introdução

In [ ]:
import json
import os
import random
import re
import time

import folium
import geopandas as gpd

import pandas as pd
import seaborn as sns
from bs4 import BeautifulSoup
from folium import plugins
from folium.plugins import DualMap
from osgeo import gdal, osr
from paths import input_path
from tqdm.notebook import tqdm, trange

In [ ]:
import numpy as np
import requests


<br>

# Layer: UGRHIs

In [ ]:
# Lê o arquivo csv com o nome dos municípios
df = pd.read_csv(    
    'https://raw.githubusercontent.com/michelmetran/sp/main/data/tabs/tab_municipio_ugrhi.csv',
)

# Lê o arquivo csv com o nome dos municípios
gdf = gpd.read_file(
    'https://raw.githubusercontent.com/michelmetran/sp/main/data/shps/sp_250k_wgs84.geojson',
)
gdf.drop(['municipio_nome'], axis=1, inplace=True)
gdf['id_municipio'] = gdf['id_municipio'].astype(int)
gdf['geometry'] = gdf.simplify(0.0015)

# Merge
gdf = gdf.merge(
    df,
    on='id_municipio',
    how='left'
)

# Save geojson
gdf.to_file(
    os.path.join(shps_path, 'sp_ugrhi.geojson'),
    driver='GeoJSON',
    encoding='utf-8'
)

# Results
gdf.head()

In [ ]:
# Seleciona colunas
df_ugrhi = gdf[['id_municipio', 'nome_ugrhi']].copy()

# Salva Tabela
df_ugrhi.to_csv(
    os.path.join(tabs_path, 'tab_municipio_ugrhi.csv'),
    index=False,
)

In [ ]:
def layer_ugrhi(input_geojson, m):
    # 
    gdf = gpd.read_file(input_geojson)

    # Column with category
    col_categories = 'nome_ugrhi'

    # Set palette
    palette_polygon = 'Paired'

    # Get list of unique values
    categories = set(gdf[col_categories])
    categories = list(categories)
    categories.sort()
    
    # See the palette chosed
    pal = sns.color_palette(palette_polygon, n_colors=len(categories))

    # Set dictionary
    color_polygon = dict(zip(categories, pal.as_hex()))
    
    
    stripes = plugins.pattern.StripePattern(
        angle=-45
    )
    stripes.add_to(m)    
    
    # 
    folium.GeoJson(
        gdf,        
        name='UGRHIs',
        smooth_factor=1.0,
        zoom_on_click=False,
        show=False,
        embed=False,
        style_function=lambda x: {
            'fillColor': color_polygon[x['properties'][col_categories]],
            'color': color_polygon[x['properties'][col_categories]],
            'weight': 0.5,
            'fillOpacity': 0.2,
            #'fillPattern': stripes,
        },
        highlight_function=lambda x: {
            'weight': 2,
            'fillOpacity': 0.6,
        },
        tooltip=folium.features.GeoJsonTooltip(
            fields=['municipio_nome', 'nome_ugrhi'],
            aliases=['Munícipio', 'UGRHI'],
            sticky=True,
            opacity=0.9,
            direction='right',
        ),
    #     popup=folium.GeoJsonPopup(
    #         ['popup'],
    #         parse_html=False,
    #         max_width='400',
    #         show=False,
    #         labels=False,
    #         sticky=True,            
    #     )        
    ).add_to(m)
    return m

<br>

# Layer: RMs

In [ ]:
# Lê o arquivo csv com o nome dos municípios
df = pd.read_csv(    
    'https://raw.githubusercontent.com/michelmetran/sp/main/data/tabs/tab_rms.csv',
)

# Lê o arquivo csv com o nome dos municípios
gdf = gpd.read_file(
    'https://raw.githubusercontent.com/michelmetran/sp/main/data/shps/sp_250k_wgs84.geojson',
)
gdf.drop(['municipio_nome'], axis=1, inplace=True)
gdf['id_municipio'] = gdf['id_municipio'].astype(int)
gdf['geometry'] = gdf.simplify(0.0015)

# Merge
gdf = gdf.merge(
    df,
    on='id_municipio',
    how='right'
)

# Save geojson
gdf.to_file(
    os.path.join(shps_path, 'sp_rms.geojson'),
    driver='GeoJSON',
    encoding='utf-8'
)

# Results
gdf.head()

In [ ]:
df_rm = gdf[['id_municipio', 'nome_rm']].copy()
df_rm.to_csv(
    os.path.join(tabs_path, 'tab_municipio_rm.csv'),
    index=False,
)

In [ ]:
def layer_rms(input_geojson, m):
    # 
    gdf = gpd.read_file(input_geojson)

    # Column with category
    col_categories = 'nome_rm'

    # Set palette
    palette_polygon = 'Paired'

    # Get list of unique values
    categories = set(gdf[col_categories])
    categories = list(categories)
    categories.sort()

    # See the palette chosed    
    pal = sns.dark_palette('#808080', reverse=True, as_cmap=False, n_colors=len(categories))

    # Set dictionary
    color_polygon = dict(zip(categories, pal.as_hex()))
    
    stripes = plugins.pattern.StripePattern(
        angle=-45
    )
    stripes.add_to(m)
    
    # 
    folium.GeoJson(
        gdf,
        name='RMs e AUs',
        smooth_factor=1.0,
        zoom_on_click=False,
        show=False,
        embed=False,
        style_function=lambda x: {
            'fillColor': color_polygon[x['properties'][col_categories]],
            'color': color_polygon[x['properties'][col_categories]],
            'weight': 2,
            'fillOpacity': 0.3,
            'fillPattern': stripes,
        },
        highlight_function=lambda x: {
            'weight': 3,
            'fillOpacity': 0.6,
        },
        tooltip=folium.features.GeoJsonTooltip(
            fields=['municipio_nome', 'nome_rm'],
            aliases=['Munícipio', 'RM|AU'],
            sticky=True,
            opacity=0.9,
            direction='right',
        ),
    #     popup=folium.GeoJsonPopup(
    #         ['popup'],
    #         parse_html=False,
    #         max_width='400',
    #         show=False,
    #         labels=False,
    #         sticky=True,            
    #     )        
    ).add_to(m)
    return m

<br>

# Layer: URAEs

In [ ]:
# Lê o arquivo csv com o nome dos municípios
df = pd.read_csv(
    'https://raw.githubusercontent.com/michelmetran/pl251/main/data/tabs/tab_municipio_pl251.csv',
    #os.path.join('data', 'tabs', 'tab_municipio_pl251.csv'),
)

# Lê o arquivo csv com o nome dos municípios
gdf = gpd.read_file(
    'https://raw.githubusercontent.com/michelmetran/sp/main/data/shps/sp_250k_wgs84.geojson',
)
gdf.drop(['municipio_nome'], axis=1, inplace=True)
gdf['id_municipio'] = gdf['id_municipio'].astype(int)
gdf['geometry'] = gdf.simplify(0.0015)

# Merge
gdf = gdf.merge(
    df,
    on='id_municipio',
    how='left'
)

# Results
gdf.head()

<br>

## Create pop up's column

In [ ]:
# Add Field
def popup_html(row):
    html = """
    <div>
    <p><b>{}</b> pertence à:
    <h4><b>{}</b></h4></p>
    </div>
    """.format(
        '' if pd.isnull(row['municipio_nome']) else '{}'.format(row['municipio_nome']),
        '' if pd.isnull(row['unidade'])        else '{}'.format(row['unidade']),
    )
    
    html = html.replace('\n','')
    html = re.sub('\s\s+' , ' ', html) # Remove Espaços no meio
    html = html.strip()
    return html

In [ ]:
# Calculate PopUps
gdf['popup'] = gdf.apply(popup_html, axis=1)

<br>

## Adjust Table

In [ ]:
# Delete Columns
gdf.drop(['id'], axis=1, inplace=True)
print(gdf.columns)

In [ ]:
# Save geojson
gdf.to_file(
    os.path.join(shps_path, 'sp_urae.geojson'),
    driver='GeoJSON',
    encoding='utf-8'
)

# Results
gdf.head()

In [ ]:
def layer_urae(input_geojson, m):    
    gdf = gpd.read_file(input_geojson)
    
    # Column with category
    col_categories = 'unidade'
    
    # Set palette
    palette_polygon = 'Paired'

    # Get list of unique values
    categories = set(gdf[col_categories])
    categories = list(categories)
    categories.sort()

    # See the palette chosed
    pal = sns.color_palette(palette_polygon, n_colors=len(categories))
    
    # Set dictionary
    color_polygon = dict(zip(categories, pal.as_hex()))
    color_polygon['URAE 1 - Sudeste'] = '#0505B4'
    color_polygon['URAE 2 - Centro'] = '#FF2E2F'
    color_polygon['URAE 3 - Leste'] = '#FEFF01'
    color_polygon['URAE 4 - Norte'] = '#31B505'    
    
    # dddd
    folium.GeoJson(
        gdf,
        name='URAEs',
        smooth_factor=1.0,
        zoom_on_click=False,
        embed=False,
        show=True,
        style_function=lambda x: {
            'fillColor': color_polygon[x['properties'][col_categories]],
            'color': color_polygon[x['properties'][col_categories]],
            'weight': 0.3,
            'fillOpacity': 0.3,
        },
        highlight_function=lambda x: {
            'weight': 2,
            'fillOpacity': 0.6,
        },
        tooltip=folium.features.GeoJsonTooltip(
            fields=['municipio_nome', 'unidade'],
            aliases=['Munícipio', 'Unidade'],
            sticky=True,
            opacity=0.9,
            direction='right',
        ),
        popup=folium.GeoJsonPopup(
            ['popup'],
            parse_html=False,
            max_width='400',
            show=False,
            labels=False,
            sticky=True,            
        )        
    ).add_to(m)
    
    return m

<br>

## Get Centroid

In [ ]:
def get_centroid(gdf):
    gdf['apenasparacentroid'] = 35
    gdf_dissolve = gdf.dissolve(by='apenasparacentroid')
    gdf_centroid = gdf_dissolve.representative_point()
    gdf = gdf.drop('apenasparacentroid', axis=1)
    return [float(gdf_centroid.y), float(gdf_centroid.x)]

In [ ]:
list_centroid = get_centroid(gdf)
list_centroid

<br>

# Folium Map

In [ ]:
def map_bomb(bbox):
    # Create Map
    m = folium.Map(
        location=[-22.545968889465207, -49.56185387118866],
        zoom_start=6,
        min_zoom=6,
        max_zoom=11,
        max_bounds=True,
        min_lon = bbox['min_lon']*(101/100),
        max_lon = bbox['max_lon']*(99/100),        
        min_lat = bbox['min_lat']*(101/100),
        max_lat = bbox['max_lat']*(99/100),
        #zoom_delta=0.1,
        tiles=None
    )
    
    # Add Base Map
    folium.TileLayer(
        'cartodbpositron',
        name='BaseMap',
        control=False,
    ).add_to(m)

    # Add Layers
    layer_urae(os.path.join(shps_path, 'sp_urae.geojson'), m)
    layer_rms(os.path.join(shps_path, 'sp_rms.geojson'), m)
    layer_ugrhi(os.path.join(shps_path, 'sp_ugrhi.geojson'), m)

    # Plugins
    m.fit_bounds(m.get_bounds())
    plugins.Fullscreen(
        position='topleft',
        title='Clique para Maximizar',
        title_cancel='Mininizar',
    ).add_to(m)
    folium.LayerControl('topright', collapsed=False).add_to(m)
    return m

In [ ]:
# Map without Bounds
bbox = {
    'max_lat': 0,
    'max_lon': 0,
    'min_lat': 0,
    'min_lon': 0,
}
m = map_bomb(bbox)

# Map with Bounds
bbox = {
    'max_lat': m.get_bounds()[1][0],
    'min_lat': m.get_bounds()[0][0],
    'max_lon': m.get_bounds()[1][1],
    'min_lon': m.get_bounds()[0][1],
}
m = map_bomb(bbox)

# Results
print(bbox)
m.save(os.path.join(maps_path, 'pl251_map.html'))
m

<br>

# Dual Map

In [ ]:
m = DualMap(
    location=[-22.545968889465207, -49.56185387118866],
    zoom_start=6,
    min_zoom=6,
    max_zoom=11,
    max_bounds=True,
    min_lon = bbox['min_lon']*(101/100),
    max_lon = bbox['max_lon']*(99/100),        
    min_lat = bbox['min_lat']*(101/100),
    max_lat = bbox['max_lat']*(99/100),
    tiles=None    
)

# Add Base Map
folium.TileLayer(
    'cartodbpositron',
    name='BaseMap',
    control=False,
).add_to(m.m1)

# Add Base Map
folium.TileLayer(
    'cartodbpositron',
    name='BaseMap',
    control=False,
).add_to(m.m2)

# Add Layers
layer_urae(os.path.join(shps_path, 'sp_urae.geojson'), m.m1)
layer_rms(os.path.join(shps_path, 'sp_rms.geojson'), m.m2)
layer_ugrhi(os.path.join(shps_path, 'sp_ugrhi.geojson'), m.m2)

plugins.Fullscreen(
    position='topleft',
    title='Clique para Maximizar',
    title_cancel='Mininizar',
).add_to(m)
folium.LayerControl('topright', collapsed=False).add_to(m.m2)
m.save(os.path.join(maps_path, 'pl251_map_dual.html'))
m